In [1]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

os.chdir('/content')
!rm -rf NLP_Course
REPO_URL = "https://github.com/dmytroslav/NLP_Course.git"
!git clone $REPO_URL

data_path = '/content/NLP_Course/data/processed_v2.csv'
df = pd.read_csv(data_path)

# Беремо колонку 'text' і заповнюємо порожні значення
df['text'] = df['text'].fillna('')
print(f"Розмір корпусу до фільтрації: {len(df)}")

# Фільтрація: залишаємо тільки тексти, де 3 або більше слів
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df_filtered = df[df['word_count'] >= 3].copy()
print(f"Розмір корпусу після фільтрації: {len(df_filtered)}")

# Витягуємо тексти у список для векторизаторів
documents = df_filtered['text'].tolist()

# 2. Налаштування векторизаторів згідно з ЛР8
# Для моделі LSA (використовуємо TF-IDF)
tfidf_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.90
)
X_tfidf = tfidf_vectorizer.fit_transform(documents)

# Для моделі LDA (використовуємо звичайний Count)
count_vectorizer = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.90
)
X_count = count_vectorizer.fit_transform(documents)

print(f"Розмірність матриці TF-IDF (для LSA): {X_tfidf.shape}")
print(f"Розмірність матриці Count (для LDA): {X_count.shape}")

Cloning into 'NLP_Course'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 159 (delta 63), reused 125 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 1.98 MiB | 8.33 MiB/s, done.
Resolving deltas: 100% (63/63), done.
Розмір корпусу до фільтрації: 10735
Розмір корпусу після фільтрації: 10734
Розмірність матриці TF-IDF (для LSA): (10734, 4337)
Розмірність матриці Count (для LDA): (10734, 4337)


In [2]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
import numpy as np


n_topics_list = [5, 8]
n_top_words = 10 # Виводимо топ-10 слів

def print_top_words(model, feature_names, n_top_words, model_name, k):
    print(f"\n{'='*50}")
    print(f"Модель: {model_name} | Кількість тем (k): {k}")
    print(f"{'='*50}")
    for topic_idx, topic in enumerate(model.components_):
        # Знаходимо індекси слів з найбільшою вагою у темі
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(f"Topic {topic_idx}: {', '.join(top_features)}")

# Отримуємо словники токенів з наших векторизаторів
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
count_feature_names = count_vectorizer.get_feature_names_out()

document_topic_matrices = {}
trained_models = {}

for k in n_topics_list:
    # 1. Варіант 1: LSA (TruncatedSVD + TF-IDF)
    lsa_model = TruncatedSVD(n_components=k, random_state=42)
    lsa_W = lsa_model.fit_transform(X_tfidf)

    document_topic_matrices[f'LSA_{k}'] = lsa_W
    trained_models[f'LSA_{k}'] = lsa_model
    print_top_words(lsa_model, tfidf_feature_names, n_top_words, "LSA", k)

    # 2. Варіант 2: LDA (Latent Dirichlet Allocation + Count)
    lda_model = LatentDirichletAllocation(n_components=k, random_state=42, n_jobs=-1)
    lda_W = lda_model.fit_transform(X_count)

    document_topic_matrices[f'LDA_{k}'] = lda_W
    trained_models[f'LDA_{k}'] = lda_model
    print_top_words(lda_model, count_feature_names, n_top_words, "LDA", k)


Модель: LSA | Кількість тем (k): 5
Topic 0: на, та, про, україни, до, україні, не, що, це, росії
Topic 1: українські, канали, телеграм, про, повідомляють, це, змі, російські, вибухи, пишуть
Topic 2: на, зсу, україни, окупанти, території, області, херсонщині, із, окупантів, напрямку
Topic 3: на, україні, сша, про, допомоги, українські, канали, повідомляють, військової, це
Topic 4: по, україні, області, удару, окупанти, завдали, зсу, ракетного, ударів, сша

Модель: LDA | Кількість тем (k): 5
Topic 0: не, до, та, україні, україна, що, україни, росії, для, через
Topic 1: на, та, україні, україни, до, про, для, із, росії, від
Topic 2: на, україні, за, україни, що, та, сша, про, не, зеленський
Topic 3: про, на, та, телеграм, українські, це, області, канали, повідомляють, російські
Topic 4: на, під, україни, та, росії, по, зсу, із, області, час

Модель: LSA | Кількість тем (k): 8
Topic 0: на, та, про, україни, до, україні, не, що, це, росії
Topic 1: українські, канали, телеграм, про, повідом

In [3]:
# Section 8. Top documents per topic
model_name = 'LSA_8'
W_matrix = document_topic_matrices[model_name]
num_top_docs = 3

print(f"=== Top Documents для {model_name} ===")
for topic_idx in range(8):
    # Отримуємо індекси документів з найбільшою вагою для цієї теми
    top_doc_indices = W_matrix[:, topic_idx].argsort()[::-1][:num_top_docs]

    print(f"\n--- Тема {topic_idx} ---")
    # Виводимо топ-слова ще раз для зручності
    top_features_ind = trained_models[model_name].components_[topic_idx].argsort()[:-n_top_words - 1:-1]
    top_words = [tfidf_feature_names[i] for i in top_features_ind]
    print(f"Топ-слова: {', '.join(top_words)}")

    # Виводимо самі тексти
    for i, doc_idx in enumerate(top_doc_indices):
        print(f"  Документ {i+1} (Вага: {W_matrix[doc_idx, topic_idx]:.3f}):")
        print(f"  {documents[doc_idx][:300]}...\n")

=== Top Documents для LSA_8 ===

--- Тема 0 ---
Топ-слова: на, та, про, україни, до, україні, не, що, це, росії
  Документ 1 (Вага: 0.427):
  Російські військові задіяли "Катюші" у спецоперації на території України. Про це пишуть українські телеграм-канали....

  Документ 2 (Вага: 0.383):
  Хвилина імпортозаміщення на болотах: голлівудські фільми замінять індійськими...

  Документ 3 (Вага: 0.377):
  Володимир Путін виступить із терміновим зверненням до росіян. Про це повідомляють українські та російські телеграм-канали....


--- Тема 1 ---
Топ-слова: українські, канали, телеграм, про, повідомляють, це, російські, змі, вибухи, пишуть
  Документ 1 (Вага: 0.530):
  Українські військові захопили трофейну ЗРПК "Панцир-С1". Про це пишуть українські телеграм-канали....

  Документ 2 (Вага: 0.496):
  Українські канали повідомляють про вибухи у Києві...

  Документ 3 (Вага: 0.443):
  Росіян готують до вживання комах. Про це повідомляють українські телеграм-канали....


--- Тема 2 ---
Топ-слова

# Аналіз тематичного моделювання (Lab 8)

## 1. Модель і параметри
Для остаточного аналізу обрано модель **LSA (TruncatedSVD)** з кількістю тем **k=8**.
Векторизація проводилась за допомогою `TfidfVectorizer` (analyzer="word", ngram_range=(1, 1), min_df=5, max_df=0.90). Ця модель показала найкращу здатність виділяти конкретні інформаційні сюжети з новинного корпусу порівняно з LDA.

## 2. Найкращі теми
На основі ручного аналізу топ-слів та топ-документів , було виділено кілька осмислених тем:

**Тема 4: Ракетні та артилерійські удари**
* **Топ-слова:** по, україні, області, удару, окупанти, завдали, зсу, ракетного, на, ударів.
* **Пояснення:** Ця тема дуже чітко згрупувала новинні зведення про обстріли українських міст. Топ-документи підтверджують це: вони містять інформацію про удари по Вугледару, Сумщині та Мелітополю. Модель успішно вловила специфічний контекст навколо дієслова "завдали" та іменника "удару".

**Тема 6: Оголошення повітряних тривог**
* **Топ-слова:** україни, про, по, вибухи, україні, тривога, повітряна, сша, повідомляють, території.
* **Пояснення:** Тема відображає високу частотність шаблонних повідомлень у Telegram-каналах. Усі топ-документи цієї теми містять ідентичні фрази на кшталт "Повітряна тривога оголошена по всій території України". Це дуже однорідний і чистий кластер.

**Тема 5: Економічні наслідки та міжнародна допомога**
* **Топ-слова:** та, україні, росії, сша, допомоги, для, військової, доларів, між, млн.
* **Пояснення:** Тема об'єднує сюжети про фінансову підтримку України (пакети допомоги від США) та економічні санкції (вихід компаній Bolt, PepsiCo з ринку РФ). Це показує, що алгоритм здатен групувати тексти за ширшим контекстом "гроші/санкції/міжнародні відносини".

## 3. Погані теми
Не всі виділені кластери виявилися корисними. Знайдено кілька проблемних тем :

**Тема 0: Stop-word topic**
* **Чому погана:** Тема складається виключно зі службових слів та прийменників ("на", "та", "про", "до", "не", "що"). Вона не несе жодного семантичного змісту і є результатом того, що ці слова найчастіше зустрічаються в текстах.
* **Що змінити:** Потрібно додати кастомний список українських стоп-слів у `TfidfVectorizer` під час preprocessing, щоб виключити цю лексику з побудови матриці.

**Тема 7: Mixed / Generic topic**
* **Чому погана:** Тема змішує абсолютно різні сюжети. Топ-слова вказують на загальну термінологію новин ("телеграм", "канали", "змі"), а в топ-документах лежать повідомлення про мобілізацію в Пітері та заборону в'їзду блогерам. Модель згрупувала їх просто через наявність стандартної новинної лексики .
* **Що змінити:** Варто перейти на лематизований текст (`lemma_text`) та збільшити `ngram_range` до (1, 2) або (1, 3), щоб алгоритм фокусувався на стійких словосполученнях, а не на окремих популярних словах.

## 4. Порівняння LSA vs LDA
Загалом модель **LSA** дала значно більш читабельні та конкретні теми порівняно з LDA . Модель LDA на нашому корпусі створила дуже змішані теми, кожна з яких була сильно "забруднена" стоп-словами ("на", "та", "про"), через що виділити унікальні сюжети було майже неможливо . LSA виявилася кориснішою для нашого кейсу, оскільки використання TF-IDF краще штрафує занадто часті слова і дозволяє витягнути більш специфічні терміни, такі як "ракетний удар" або "повітряна тривога" .

## 5. Висновок
Тематичне моделювання показало, що наш корпус містить чітко виражені новинні кластери (обстріли, тривоги, міжнародна політика). Однак наявна значна кількість технічного шуму та стоп-слів, що вимагає більш агресивного очищення тексту перед використанням unsupervised методів.